# Shrub Lists Extras — To Targets

Convert standardized shrub lists into training-ready targets: point-level tables and grid-based target rasters.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
STD_ORIG = Path('standardized') / 'shrubs_original_standardized.csv'
STD_REV  = Path('standardized') / 'shrubs_revised_standardized.csv'

shrubs_orig = pd.read_csv(STD_ORIG)
shrubs_rev  = pd.read_csv(STD_REV)


In [ ]:
def make_consensus_labels(orig, rev, *, tol_xy=1.0):
    if not {'x','y'}.issubset(orig.columns) or not {'x','y'}.issubset(rev.columns):
        raise ValueError('Need x/y columns in both dataframes.')

    a = orig[['x','y']].to_numpy(dtype=float)
    b = rev[['x','y']].to_numpy(dtype=float)
    keep_pairs = []

    for j, (x2, y2) in enumerate(b):
        d2 = (a[:,0] - x2)**2 + (a[:,1] - y2)**2
        i = int(np.argmin(d2))
        if np.sqrt(d2[i]) <= tol_xy:
            keep_pairs.append((i, j))

    if not keep_pairs:
        return pd.DataFrame(columns=orig.columns)

    orig_idx = sorted(set(i for i, _ in keep_pairs))
    cons = orig.iloc[orig_idx].copy().reset_index(drop=True)
    cons['label_source'] = 'consensus_original_revised'
    return cons

consensus = make_consensus_labels(shrubs_orig, shrubs_rev, tol_xy=1.0)
print('consensus size:', len(consensus))
consensus.head()


In [ ]:
def rasterize_shrub_points(df, *, bbox, cell_size=1.0, x_col='x', y_col='y', height_col='height_m'):
    xmin, ymin, xmax, ymax = bbox
    w = int(np.ceil((xmax - xmin) / cell_size))
    h = int(np.ceil((ymax - ymin) / cell_size))

    cols = [x_col, y_col] + ([height_col] if height_col in df.columns else [])
    work = df[cols].dropna(subset=[x_col, y_col]).copy()
    work = work[(work[x_col] >= xmin) & (work[x_col] < xmax) & (work[y_col] >= ymin) & (work[y_col] < ymax)].copy()

    col = np.floor((work[x_col].to_numpy() - xmin) / cell_size).astype(int)
    row = np.floor((work[y_col].to_numpy() - ymin) / cell_size).astype(int)

    count = np.zeros((h, w), dtype=np.int32)
    presence = np.zeros((h, w), dtype=np.uint8)
    height_sum = np.zeros((h, w), dtype=np.float32)
    height_n = np.zeros((h, w), dtype=np.int32)

    has_h = height_col in work.columns
    heights = work[height_col].to_numpy(dtype=float) if has_h else None

    for k in range(len(work)):
        r, c = row[k], col[k]
        if 0 <= r < h and 0 <= c < w:
            count[r, c] += 1
            presence[r, c] = 1
            if has_h and np.isfinite(heights[k]):
                height_sum[r, c] += heights[k]
                height_n[r, c] += 1

    mean_height = np.full((h, w), np.nan, dtype=np.float32)
    mask = height_n > 0
    mean_height[mask] = height_sum[mask] / height_n[mask]

    return {
        'presence': presence,
        'count': count,
        'mean_height': mean_height,
        'bbox': bbox,
        'cell_size': cell_size,
        'shape': (h, w),
    }


In [ ]:
base_df = consensus if len(consensus) > 0 else shrubs_rev
xmin = float(base_df['x'].min()); ymin = float(base_df['y'].min())
xmax = float(base_df['x'].max()); ymax = float(base_df['y'].max())
pad = 5.0
bbox_proj = (xmin - pad, ymin - pad, xmax + pad, ymax + pad)

grid = rasterize_shrub_points(base_df, bbox=bbox_proj, cell_size=1.0)
print(grid['shape'], grid['bbox'])


In [ ]:
plt.figure()
plt.imshow(grid['presence'], origin='lower')
plt.title('Shrub presence grid')
plt.colorbar()
plt.show()

plt.figure()
plt.imshow(grid['count'], origin='lower')
plt.title('Shrub count grid')
plt.colorbar()
plt.show()
